# 🧠 Band-Power SVM — Motor Imagery EEG
All subjects × all sessions × LORO-CV. No GPU needed.
Replicates the paper's method. Checkpoint: `checkpoints/svm_subXX_sesXX_runX.pkl`

## 1. Imports & config

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import mne
import torch
import torch.nn as nn
import torch.nn.functional as torch_func
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
from scipy.signal import butter, sosfilt, sosfilt_zi
from collections import deque
import joblib, time
warnings.filterwarnings("ignore")
mne.set_log_level("WARNING")

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

def get_device():
    try:
        import torch_directml
        dev = torch_directml.device(0)
        print(f"✅ AMD GPU: {torch_directml.device_name(0)}"); return dev
    except: pass
    if torch.cuda.is_available():
        print(f"✅ CUDA: {torch.cuda.get_device_name(0)}"); return torch.device("cuda")
    print("⚠️  CPU only"); return torch.device("cpu")

DEVICE = get_device()
ROOT   = os.getcwd()
N_SUBJECTS = 11
N_SESSIONS = {i: 6 for i in range(1,12)}; N_SESSIONS[2] = 4
N_RUNS = 3; SFREQ = 256
EEG_CHANNELS = ["F3","F1","Fz","F2","F4","C3","C1","Cz","C2","C4",
                 "CPz","P3","P1","Pz","P2","P4","POz","O1","Oz","O2"]
FMIN, FMAX = 8.0, 30.0
TMIN, TMAX = 0.5, 4.5
print(f"Root: {ROOT} | Config ready ✅")

## 2. Data loader

In [ ]:
def build_path(root, sub, ses, run, suffix):
    fname = f"sub-{sub:02d}_ses-{ses:02d}_task-MI_run-{run:02d}_{suffix}"
    return os.path.join(root, f"sub-{sub:02d}", f"ses-{ses:02d}", "eeg", fname)

def load_session(root, sub, ses):
    """Returns (runs_X, runs_y): list of N_RUNS arrays, None where file missing."""
    runs_X, runs_y = [], []
    for run in range(1, N_RUNS+1):
        edf = build_path(root, sub, ses, run, "eeg.edf")
        ev  = build_path(root, sub, ses, run, "events.tsv")
        if not os.path.exists(edf):
            runs_X.append(None); runs_y.append(None); continue
        raw = mne.io.read_raw_edf(edf, preload=True, verbose=False)
        avail = [c for c in EEG_CHANNELS if c in raw.ch_names]
        raw.pick(avail)
        raw.filter(FMIN, FMAX, method="iir",
                   iir_params=dict(order=4, ftype="butter"), verbose=False)
        ev_df  = pd.read_csv(ev, sep="\t")
        evs    = np.array([[int(r["sample"]),0,1 if r["trial_type"]=="action" else 0]
                            for _,r in ev_df.iterrows()])
        evs    = evs[np.argsort(evs[:,0])]
        epochs = mne.Epochs(raw, evs, event_id={"rest":0,"action":1},
                             tmin=TMIN, tmax=TMAX, baseline=None,
                             preload=True, verbose=False)
        X = epochs.get_data(); y = epochs.events[:,2].astype(np.int64)
        keep = np.max(np.abs(X), axis=(1,2)) < 100e-6
        X, y = X[keep], y[keep]
        mu = X.mean(axis=2, keepdims=True); std = X.std(axis=2, keepdims=True)+1e-8
        runs_X.append((X-mu)/std); runs_y.append(y)
    return runs_X, runs_y

print("Data loader ready ✅")

## 3. Feature extraction & SVM pipeline

In [ ]:
from scipy.signal import welch as _welch
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

MI_CHANNELS = ["C3","C1","Cz","C2","C4"]
FREQ_BINS   = [(6,11),(11,16),(16,21),(21,26),(26,31),(31,36)]

def extract_bandpower(X, ch_names):
    """X: (n_epochs, n_ch, n_t) → (n_epochs, n_features)"""
    mi_idx=[ch_names.index(c) for c in MI_CHANNELS if c in ch_names]
    feats=[]
    for epoch in X:
        ep_f=[]
        for ci in mi_idx:
            f,psd=_welch(epoch[ci],fs=SFREQ,nperseg=SFREQ//2)
            for flo,fhi in FREQ_BINS:
                mask=(f>=flo)&(f<fhi); ep_f.append(np.log(psd[mask].mean()+1e-10))
        feats.append(ep_f)
    return np.array(feats)

def build_svm():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("svm",    SVC(kernel="rbf", C=1.0, gamma="scale", probability=True)),
    ])

CKPT_PREFIX = "svm"
print(f"Feature dim: {len(MI_CHANNELS)*len(FREQ_BINS)} | SVM ready ✅")

## 4. Train — all subjects × all sessions

In [ ]:
SUBJECTS = list(range(1, 12))   # ← e.g. [1,2,3] for quick test
os.makedirs("checkpoints", exist_ok=True)
all_rows = []

for sub in SUBJECTS:
    print(f"\n{'='*55}\n  sub-{sub:02d}  ({N_SESSIONS[sub]} sessions)\n{'='*55}")
    for ses in range(1, N_SESSIONS[sub]+1):
        print(f"  ── ses-{ses:02d} ──")
        runs_X, runs_y = load_session(ROOT, sub, ses)
        valid = [i for i in range(N_RUNS) if runs_X[i] is not None and len(runs_X[i])>0]
        if len(valid) < 2:
            print(f"    ⚠️  Not enough runs — skipping"); continue
        _edf = build_path(ROOT, sub, ses, valid[0]+1, "eeg.edf")
        _raw = mne.io.read_raw_edf(_edf, preload=False, verbose=False)
        ch_names = [c for c in EEG_CHANNELS if c in _raw.ch_names]; del _raw
        for fold in range(N_RUNS):
            if runs_X[fold] is None or len(runs_X[fold])==0: continue
            X_te=runs_X[fold]; y_te=runs_y[fold]
            tr_X=[runs_X[i] for i in range(N_RUNS) if i!=fold and runs_X[i] is not None and len(runs_X[i])>0]
            tr_y=[runs_y[i] for i in range(N_RUNS) if i!=fold and runs_y[i] is not None and len(runs_y[i])>0]
            if not tr_X: continue
            X_tr=np.concatenate(tr_X); y_tr=np.concatenate(tr_y)
                        feat_tr=extract_bandpower(X_tr,ch_names); feat_te=extract_bandpower(X_te,ch_names)
            pipe=build_svm()
            t0=time.perf_counter(); pipe.fit(feat_tr,y_tr); elapsed=time.perf_counter()-t0
            preds=pipe.predict(feat_te)
            acc=accuracy_score(y_te,preds); bal=balanced_accuracy_score(y_te,preds)
            print(f"    run-{fold+1} | acc={acc:.3f}  bal={bal:.3f}  ({elapsed*1000:.0f} ms)")
            ckpt=f"checkpoints/{CKPT_PREFIX}_sub{sub:02d}_ses{ses:02d}_run{fold+1}.pkl"
            joblib.dump({"pipe":pipe,"ch_names":ch_names},ckpt)
            all_rows.append({"subject":sub,"session":ses,"run":fold+1,"acc":acc,"bal_acc":bal})
        ses_r=[r for r in all_rows if r["subject"]==sub and r["session"]==ses]
        if ses_r: print(f"  ses-{ses:02d} mean → acc={np.mean([r['acc'] for r in ses_r]):.3f}  bal={np.mean([r['bal_acc'] for r in ses_r]):.3f}")
    sub_r=[r for r in all_rows if r["subject"]==sub]
    if sub_r: print(f"\n  sub-{sub:02d} OVERALL → acc={np.mean([r['acc'] for r in sub_r]):.3f}  bal={np.mean([r['bal_acc'] for r in sub_r]):.3f}")

results_df=pd.DataFrame(all_rows)
print(f"\n✅ Done — {len(results_df)} folds | {results_df['subject'].nunique()} subjects | {results_df['session'].nunique()} sessions")

## 5. Summary plot & table

In [ ]:
def plot_summary(results_df, method_name, out_file):
    sub_mean = results_df.groupby("subject")[["acc","bal_acc"]].mean().reset_index()
    fig, axes = plt.subplots(1, 2, figsize=(16,5))
    fig.suptitle(f"{method_name} — Results Summary", fontweight="bold", fontsize=14)

    ax = axes[0]; x = np.arange(len(sub_mean)); w = 0.35
    ax.bar(x-w/2, sub_mean["acc"],     w, label="Accuracy",          color="#4C9BE8", alpha=0.85)
    ax.bar(x+w/2, sub_mean["bal_acc"], w, label="Balanced accuracy", color="#E8754C", alpha=0.85)
    ax.axhline(0.50, color="red",   ls="--", lw=1.2, label="Chance (50%)")
    ax.axhline(0.75, color="green", ls="--", lw=1.2, label="Literature (75%)")
    ax.set_xticks(x)
    ax.set_xticklabels([f"sub-{int(s):02d}" for s in sub_mean["subject"]], rotation=45, ha="right")
    ax.set_ylim(0,1); ax.set_ylabel("Score"); ax.set_title("Mean accuracy per subject"); ax.legend(fontsize=8)

    ax2 = axes[1]
    pivot = results_df.groupby(["subject","session"])["acc"].mean().unstack(fill_value=np.nan)
    im = ax2.imshow(pivot.values, aspect="auto", cmap="RdYlGn", vmin=0.5, vmax=1.0)
    ax2.set_xticks(range(pivot.shape[1]))
    ax2.set_xticklabels([f"ses-{int(c):02d}" for c in pivot.columns])
    ax2.set_yticks(range(pivot.shape[0]))
    ax2.set_yticklabels([f"sub-{int(r):02d}" for r in pivot.index])
    plt.colorbar(im, ax=ax2, label="Accuracy"); ax2.set_title("Accuracy heatmap")
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i,j]
            if not np.isnan(v): ax2.text(j,i,f"{v:.2f}",ha="center",va="center",fontsize=7)
    plt.tight_layout(); plt.savefig(out_file, dpi=150); plt.show()

    print(f"\n{'Sub':<8} {'Sessions':>9} {'Mean Acc':>10} {'Bal Acc':>10}")
    print("─"*42)
    for _, row in sub_mean.iterrows():
        n = len(results_df[results_df["subject"]==row["subject"]]["session"].unique())
        print(f"sub-{int(row['subject']):02d}   {n:>9}  {row['acc']:>10.3f} {row['bal_acc']:>10.3f}")
    print("─"*42)
    print(f"{'MEAN':<8} {'':>9}  {sub_mean['acc'].mean():>10.3f} {sub_mean['bal_acc'].mean():>10.3f}")
    print(f"{'STD':<8} {'':>9}  {sub_mean['acc'].std():>10.3f}  {sub_mean['bal_acc'].std():>10.3f}")
    results_df.to_csv(out_file.replace(".png",".csv"), index=False)
    print(f"Saved → {out_file.replace('.png','.csv')}")

plot_summary(results_df, 'Band-Power SVM', 'svm_summary.png')

## 6. Real-time — all subjects × all sessions
> **Run-3 checkpoint** trained on runs 1+2 only → genuinely unseen ✅

In [ ]:
METHOD_NAME='Band-Power SVM'
CKPT_PREFIX='svm'
CKPT_EXT='pkl'

def LOAD_FN(ckpt_path): return joblib.load(ckpt_path)

def INFER_FN(pipe_data, X_rt, ch_names):
    feats=extract_bandpower(X_rt[np.newaxis], pipe_data['ch_names'])
    return pipe_data['pipe'].predict_proba(feats)[0,1]


def run_realtime_all(subjects, rt_run=3, threshold=0.55):
    """
    Loop all subjects x all sessions. For each:
      - load checkpoint trained WITHOUT rt_run  → genuinely unseen data ✅
      - stream that run sample-by-sample
      - compute online accuracy and save figure
    """
    os.makedirs("realtime_figures", exist_ok=True)
    rt_rows = []

    for sub in subjects:
        for ses in range(1, N_SESSIONS[sub]+1):
            ckpt = f"checkpoints/{CKPT_PREFIX}_sub{sub:02d}_ses{ses:02d}_run{rt_run}.{CKPT_EXT}"
            if not os.path.exists(ckpt):
                continue

            edf_path    = build_path(ROOT, sub, ses, rt_run, "eeg.edf")
            events_path = build_path(ROOT, sub, ses, rt_run, "events.tsv")
            if not os.path.exists(edf_path):
                continue

            print(f"  sub-{sub:02d} ses-{ses:02d} run-{rt_run}…", end=" ", flush=True)

            # Load model / pipeline
            model_or_pipe = LOAD_FN(ckpt)

            # Load EEG
            raw_rt = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
            avail  = [c for c in EEG_CHANNELS if c in raw_rt.ch_names]
            raw_rt.pick(avail)
            data   = raw_rt.get_data()
            n_ch, n_samples = data.shape
            times  = np.arange(n_samples) / SFREQ

            ev_rt = pd.read_csv(events_path, sep="\t")
            gt    = np.full(n_samples, -1, dtype=int)
            for _, row in ev_rt.iterrows():
                s = int(row["sample"]); d = int(row["duration"]*SFREQ)
                gt[s:s+d] = 1 if row["trial_type"]=="action" else 0

            sos     = butter(4,[FMIN,FMAX],btype="bandpass",fs=SFREQ,output="sos")
            zi      = np.stack([sosfilt_zi(sos)]*n_ch, axis=1)
            epoch_s = int(4.0*SFREQ); step_s = int(0.25*SFREQ)
            buffer  = np.zeros((n_ch,epoch_s), dtype=np.float32)
            prob_q  = deque(maxlen=5)
            decisions, buf_count = [], 0

            for i in range(n_samples):
                for c in range(n_ch):
                    out, zi[:,c] = sosfilt(sos, data[c:c+1,i], zi=zi[:,c])
                    buffer[c]    = np.roll(buffer[c],-1); buffer[c,-1] = out[0]
                buf_count += 1
                if i >= epoch_s and buf_count >= step_s:
                    buf_count = 0
                    X_rt = buffer.copy()
                    X_rt = (X_rt-X_rt.mean(1,keepdims=True))/(X_rt.std(1,keepdims=True)+1e-8)
                    p    = INFER_FN(model_or_pipe, X_rt, avail)
                    prob_q.append(p)
                    sm   = float(np.mean(prob_q))
                    decisions.append((i, p, sm, int(sm>=threshold)))

            if not decisions: print("no windows"); continue
            dec_arr      = np.array(decisions)
            dec_samples  = dec_arr[:,0].astype(int)
            raw_probs    = dec_arr[:,1]; smooth_probs = dec_arr[:,2]
            dec_labels   = dec_arr[:,3].astype(int)
            dec_times    = dec_samples/SFREQ
            labeled      = gt[dec_samples]>=0
            online_acc   = (gt[dec_samples[labeled]]==dec_labels[labeled]).mean() if labeled.sum()>0 else None
            print(f"online acc = {online_acc*100:.1f}%" if online_acc is not None else "no labels")

            # Plot
            fig, axes = plt.subplots(3,1,figsize=(16,8),sharex=True)
            title = (f"{METHOD_NAME} — sub-{sub:02d} ses-{ses:02d} run-{rt_run}  "
                     f"(online acc={online_acc*100:.1f}%)" if online_acc else
                     f"{METHOD_NAME} — sub-{sub:02d} ses-{ses:02d} run-{rt_run}")
            fig.suptitle(title, fontsize=12, fontweight="bold")
            for ax in axes:
                for _,row in ev_rt.iterrows():
                    col="#AEDEA7" if row["trial_type"]=="action" else "#AEC6E8"
                    ax.axvspan(row["onset"],row["onset"]+row["duration"],color=col,alpha=0.3,lw=0)
            cz = avail.index("Cz") if "Cz" in avail else 0
            axes[0].plot(times,data[cz]*1e6,lw=0.4,color="#333")
            axes[0].set_ylabel("µV"); axes[0].set_title("Cz — Raw EEG")
            axes[0].legend(handles=[mpatches.Patch(color="#AEDEA7",alpha=0.5,label="Action"),
                                     mpatches.Patch(color="#AEC6E8",alpha=0.5,label="Rest")],
                            loc="upper right",fontsize=8)
            axes[1].fill_between(dec_times,raw_probs,alpha=0.3,color="#7FC8F8",label="Raw p(action)")
            axes[1].plot(dec_times,smooth_probs,lw=1.5,color="#1B6CA8",label="Smoothed")
            axes[1].axhline(threshold,color="r",ls="--",lw=1,label=f"Threshold ({threshold})")
            axes[1].set_ylim(0,1); axes[1].set_ylabel("p(action)")
            axes[1].set_title("Classification probability"); axes[1].legend(fontsize=8)
            gt_vis=np.where(gt>=0,gt.astype(float),np.nan)
            axes[2].step(dec_times,dec_labels,lw=1.5,color="#D62728",where="post",label="Prediction")
            axes[2].plot(times,gt_vis,lw=1.5,color="#2CA02C",ls="--",alpha=0.8,label="Ground truth")
            axes[2].set_ylim(-0.1,1.3); axes[2].set_yticks([0,1]); axes[2].set_yticklabels(["REST","ACTION"])
            axes[2].set_xlabel("Time (s)"); axes[2].set_title("Decision vs Ground Truth"); axes[2].legend(fontsize=8)
            plt.tight_layout()
            out = f"realtime_figures/{CKPT_PREFIX}_sub{sub:02d}_ses{ses:02d}_run{rt_run}.png"
            plt.savefig(out,dpi=100); plt.close()
            if online_acc is not None:
                rt_rows.append({"subject":sub,"session":ses,"online_acc":online_acc})

    if not rt_rows:
        print("No results — run Section 4 first."); return
    rt_df = pd.DataFrame(rt_rows)
    sub_mean = rt_df.groupby("subject")["online_acc"].mean()
    print(f"\n── Online Accuracy (run-{rt_run} held-out) ──")
    print(f"{'Sub':<8} {'Mean Online Acc':>16}"); print("─"*26)
    for sub, acc in sub_mean.items():
        print(f"sub-{sub:02d}  {acc*100:>14.1f}%")
    print("─"*26); print(f"MEAN    {sub_mean.mean()*100:>14.1f}%")
    rt_df.to_csv(f"realtime_{CKPT_PREFIX}_summary.csv",index=False)
    print(f"Saved → realtime_{CKPT_PREFIX}_summary.csv")

run_realtime_all(list(range(1,12)), rt_run=3, threshold=0.55)